# Unit 13. Unsupervised Learning

精算統計模型 (SOA Exam SRM/PA). The text of [unit13.pdf](https://github.com/chang-ye-tu/asm/blob/master/note/unit13.pdf) with every R chunk as a code cell.

<a href="https://colab.research.google.com/github/chang-ye-tu/asm/blob/master/colab/unit13.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

The notebook runs on the R runtime (on Colab: Runtime › Change runtime type › R). Run the setup cell first: it installs the packages and downloads the data this unit uses.

In [ ]:
# Setup: run this cell first. It installs the packages this unit uses and downloads
# its data files into data/, the paths the code below reads.
local({
  if (is.null(getOption("repos")) || identical(unname(getOption("repos")["CRAN"]), "@CRAN@"))
    options(repos = c(CRAN = "https://cloud.r-project.org"))
  os <- tryCatch(readLines("/etc/os-release"), error = function(e) character())
  code <- sub("^VERSION_CODENAME=", "", grep("^VERSION_CODENAME=", os, value = TRUE))
  if (length(code) == 1 && nzchar(code))  # Linux, e.g. Colab: prebuilt binary packages
    options(repos = c(CRAN = sprintf("https://packagemanager.posit.co/cran/__linux__/%s/latest", code)),
            HTTPUserAgent = sprintf("R/%s R (%s)", getRversion(), paste(getRversion(),
              R.version$platform, R.version$arch, R.version$os)))
})
pkgs <- c("ISLR2")
new <- setdiff(pkgs, rownames(installed.packages()))
if (length(new)) install.packages(new)
data_files <- c("data/islr/Ch12Ex13.csv")
for (f in data_files) if (!file.exists(f)) {
  dir.create(dirname(f), recursive = TRUE, showWarnings = FALSE)
  url <- if (startsWith(f, "data/islr/"))
    paste0("https://www.statlearning.com/s/", basename(f)) else
    paste0("https://raw.githubusercontent.com/chang-ye-tu/asm/master/note/", f)
  download.file(url, f, quiet = TRUE, mode = "wb")
}
options(width = 62, digits = 4, scipen = 4, repr.plot.width = 6.4, repr.plot.height = 3.0)

## 1 Unsupervised Learning

> **Reading.** ISLR §12.1, §12.2 (both reading lists; §12.2 is proved in Unit 5). SRM learning outcomes 5(a), 5(b).

**Remark** (ISLR §12.1). In unsupervised learning there are features $X_1,\ldots,X_p$ on $n$ observations and no response. Nothing is predicted, so there is no test error and no cross-validation: the goals are an informative picture of the data and the discovery of subgroups, among the observations or among the variables. ISLR's examples are subgroups of breast cancer patients by gene expression, groups of shoppers by browsing and purchase history, and the choice of search results to show a given user. The analysis is often part of exploratory data analysis, and there is no universally accepted way to check its result.

**Remark** (ISLR §12.2; Unit 5). Principal components were treated in Unit 5, and its results are used here by name. The loading vectors $\boldsymbol{\phi}_m$ are the eigenvectors of the sample covariance matrix $\mathbf{S}$ and the score vectors $\mathbf{z}_m=\mathbf{X}\boldsymbol{\phi}_m$ are uncorrelated, unique up to sign when the eigenvalues are distinct (Unit 5's theorem **principal components are eigenvectors**). The proportion of variance explained by the $m$th component is $\lambda_m/\sum_l\lambda_l$ (**proportion of variance explained**). The first $M$ components give the best rank-$M$ approximation of $\mathbf{X}$ and the closest $M$-dimensional subspace (**best approximation and closest subspaces**). Rescaling the columns changes the components (**scaling changes the components**), and a scree plot is read for an elbow. What remains for this unit is the biplot of ISLR's Figure 12.1, the picture in which loadings and scores are read together.

**Definition 13.1** (Biplot; ISLR Figure 12.1). The scatter of the score pairs $(z_{i1},z_{i2})$, $i=1,\ldots,n$, with the loading pairs $\mathbf{a}_j=(\phi_{j1},\phi_{j2})$, $j=1,\ldots,p$, drawn as arrows from the origin on a common pair of axes.

**Proposition 13.2** (What the arrows mean). With $\mathbf{X}$ column-centred, $\mathbf{S}=n^{-1}\mathbf{X}'\mathbf{X}$, $\boldsymbol{\phi}_m=\mathbf{v}_m$, $\lambda_m$, $\mathbf{Z}_M$ and $\boldsymbol{\Phi}_M$ as in Unit 5's definition **principal components** and theorems **principal components are eigenvectors** and **best approximation and closest subspaces**, let $\mathbf{z}_{(i)}=(z_{i1},z_{i2})$ and let $\hat x_{ij}$ be the $(i,j)$ entry of the rank-two approximation $\mathbf{Z}_2\boldsymbol{\Phi}_2'$ of $\mathbf{X}$. Then

(i) $\hat x_{ij}=\mathbf{z}_{(i)}'\mathbf{a}_j$: the reconstruction of variable $j$ for observation $i$ is the inner product of the score point with the $j$th arrow;

(ii) if $\lambda_1,\lambda_2>0$ and $\mathbf{a}_j,\mathbf{a}_k\neq\mathbf{0}$, the sample correlation of the reconstructed columns $\hat{\mathbf{x}}_j$ and $\hat{\mathbf{x}}_k$ is the cosine of the angle between $\boldsymbol{\Lambda}^{1/2}\mathbf{a}_j$ and $\boldsymbol{\Lambda}^{1/2}\mathbf{a}_k$, $\boldsymbol{\Lambda}=\operatorname{diag}(\lambda_1,\lambda_2)$. In particular two parallel arrows give perfectly correlated reconstructions, and two perpendicular arrows give uncorrelated ones when $\lambda_1=\lambda_2$.

**Proof.** (i) The $(i,j)$ entry of $\mathbf{Z}_2\boldsymbol{\Phi}_2'$ is $\sum_{m\leqslant2}z_{im}\phi_{jm}$.

(ii) By (i), $\hat{\mathbf{x}}_j=\mathbf{Z}_2\mathbf{a}_j$. The score vectors have mean zero, so the reconstructions do too, and by Unit 5's theorem **principal components are eigenvectors** $\mathbf{Z}_2'\mathbf{Z}_2=n\boldsymbol{\Lambda}$. Hence the sample covariance of $\hat{\mathbf{x}}_j$ and $\hat{\mathbf{x}}_k$ is $n^{-1}\mathbf{a}_j'\mathbf{Z}_2'\mathbf{Z}_2\mathbf{a}_k=\mathbf{a}_j'\boldsymbol{\Lambda}\mathbf{a}_k=(\boldsymbol{\Lambda}^{1/2}\mathbf{a}_j)'(\boldsymbol{\Lambda}^{1/2}\mathbf{a}_k)$, and dividing by the two standard deviations $|\boldsymbol{\Lambda}^{1/2}\mathbf{a}_j|$ and $|\boldsymbol{\Lambda}^{1/2}\mathbf{a}_k|$, both positive, gives the cosine. Parallel arrows have $\boldsymbol{\Lambda}^{1/2}\mathbf{a}_j$ and $\boldsymbol{\Lambda}^{1/2}\mathbf{a}_k$ parallel; when $\lambda_1=\lambda_2$ the matrix $\boldsymbol{\Lambda}^{1/2}$ is a multiple of $\mathbf{I}$ and preserves angles. ∎

**Remark**. So the arrows of ISLR's `USArrests` biplot (Unit 5's example `USArrests`) are read as follows. `Murder` and `Assault` point nearly together and `Rape` lies between them and `UrbanPop`, so the two-component reconstructions of the three crime rates are highly correlated ($0.80$ to $0.97$); `UrbanPop` points away from them, nearly at right angles to `Assault`. The reading concerns the reconstruction, not the data: it is as good as the two components' share of the variance, $87\%$ there.

## 2 K-means Clustering

> **Reading.** ISLR §12.4 (introduction), §12.4.1 (both reading lists). SRM learning outcome 5(c).

**Remark** (ISLR §12.4). Clustering looks for subgroups of observations that are similar within and different between. Principal components look for a low-dimensional representation that explains much of the variance; clustering looks for homogeneous groups. The observations can be clustered on the basis of the features, or the features on the basis of the observations by transposing $\mathbf{X}$. $K$-means asks for the number of clusters in advance; hierarchical clustering does not, and returns a tree from which any number can be read.

**Definition 13.3** (The criterion; ISLR (12.17)). A **clustering** into $K$ groups is a partition $C_1,\ldots,C_K$ of $\{1,\ldots,n\}$ with every $C_g$ non-empty. Its **within-cluster sum of squares** is

$$
W(C)=\sum_{g=1}^{K}\sum_{i\in C_g}\,\bigl|\mathbf{x}_i-\bar{\mathbf{x}}_g\bigr|^2 ,
\qquad \bar{\mathbf{x}}_g=\frac1{|C_g|}\sum_{i\in C_g}\mathbf{x}_i , \tag{1}
$$

and $K$-means clustering is the problem $\min_CW(C)$. Here $|C_g|$ is the number of elements of $C_g$, and for a vector $|\cdot|$ is the Euclidean norm.

**Proposition 13.4** (Three readings of the criterion). (i) $\displaystyle\sum_{i\in C_g}|\mathbf{x}_i-\bar{\mathbf{x}}_g|^2 =\frac1{2|C_g|}\sum_{i\in C_g}\sum_{i'\in C_g}|\mathbf{x}_i-\mathbf{x}_{i'}|^2$: the within-cluster sum of squares is a sum of pairwise distances and never refers to a centre.

(ii) $W(C)+B(C)=T$, where $T=\sum_i|\mathbf{x}_i-\bar{\mathbf{x}}|^2$ is free of $C$ and $B(C)=\sum_g|C_g|\,|\bar{\mathbf{x}}_g-\bar{\mathbf{x}}|^2$. Minimising $W$ is maximising $B$.

(iii) For fixed centres $\boldsymbol{\mu}_1,\ldots,\boldsymbol{\mu}_K$, $\sum_g\sum_{i\in C_g}|\mathbf{x}_i-\boldsymbol{\mu}_g|^2$ is minimised over assignments of the $n$ points to the $K$ labels (empty clusters allowed) by assigning each $i$ to a nearest centre; and for a fixed partition it is minimised over centres uniquely at $\boldsymbol{\mu}_g=\bar{\mathbf{x}}_g$.

**Proof.** (i) Expanding, $\sum_{i,i'\in C_g}|\mathbf{x}_i-\mathbf{x}_{i'}|^2 =\sum_{i,i'}\{|\mathbf{x}_i|^2-2\mathbf{x}_i'\mathbf{x}_{i'}+|\mathbf{x}_{i'}|^2\} =2|C_g|\sum_i|\mathbf{x}_i|^2-2\bigl|\sum_i\mathbf{x}_i\bigr|^2 =2|C_g|\bigl\{\sum_i|\mathbf{x}_i|^2-|C_g|\,|\bar{\mathbf{x}}_g|^2\bigr\}$, and the brace is $\sum_{i\in C_g}|\mathbf{x}_i-\bar{\mathbf{x}}_g|^2$.

(ii) For $i\in C_g$ write $\mathbf{x}_i-\bar{\mathbf{x}}=(\mathbf{x}_i-\bar{\mathbf{x}}_g)+(\bar{\mathbf{x}}_g-\bar{\mathbf{x}})$ and expand: $|\mathbf{x}_i-\bar{\mathbf{x}}|^2=|\mathbf{x}_i-\bar{\mathbf{x}}_g|^2+|\bar{\mathbf{x}}_g-\bar{\mathbf{x}}|^2 +2(\mathbf{x}_i-\bar{\mathbf{x}}_g)'(\bar{\mathbf{x}}_g-\bar{\mathbf{x}})$. Summing over $i\in C_g$ the cross term vanishes, because $\sum_{i\in C_g}(\mathbf{x}_i-\bar{\mathbf{x}}_g)=\mathbf{0}$, and summing over $g$ gives $T=W(C)+B(C)$.

(iii) The double sum splits over $i$, and the term for $i$ is $|\mathbf{x}_i-\boldsymbol{\mu}_{g(i)}|^2$, smallest when $g(i)$ indexes a nearest centre. For fixed $C$ the sum splits over $g$, and by the expansion of (ii) with $\boldsymbol{\mu}_g$ in place of $\bar{\mathbf{x}}$, $\sum_{i\in C_g}|\mathbf{x}_i-\boldsymbol{\mu}_g|^2=\sum_{i\in C_g}|\mathbf{x}_i-\bar{\mathbf{x}}_g|^2+|C_g|\,|\bar{\mathbf{x}}_g-\boldsymbol{\mu}_g|^2$, smallest exactly at $\boldsymbol{\mu}_g=\bar{\mathbf{x}}_g$. ∎

**Remark** (ISLR (12.15)–(12.18)). ISLR measures a cluster by its within-cluster variation $|C_g|^{-1}\sum_{i,i'\in C_g}|\mathbf{x}_i-\mathbf{x}_{i'}|^2$, its (12.16) with squared Euclidean distance. By Proposition 13.4(i) that is $2\sum_{i\in C_g}|\mathbf{x}_i-\bar{\mathbf{x}}_g|^2$, ISLR's (12.18), so ISLR's criterion (12.17) is $2W(C)$ and has the same minimisers. R's `kmeans` reports $W(C)$ as `tot.withinss`. By default it runs a different local search (Hartigan–Wong), which also stops at a local optimum; with `algorithm = "Lloyd"` the numbers of the simulated-data example of this unit's lab are the same.

**Definition 13.5** (Lloyd's algorithm; ISLR Algorithm 12.2). Start from an assignment; ISLR assigns each observation a cluster from $1$ to $K$ at random. Repeat: compute the cluster means; reassign each observation to a nearest mean, ties broken by a fixed rule that depends only on the means (say, the smallest cluster index). Stop when the assignment does not change. The algorithm is defined as long as no cluster becomes empty.

**Theorem 13.6** (The algorithm decreases the criterion and stops). Suppose no cluster becomes empty, and write $C^{(t)}$ for the assignment after $t$ rounds. Then $W(C^{(t+1)})\leqslant W(C^{(t)})$, and the algorithm reaches an assignment it does not change after finitely many rounds.

**Proof.** Let $\boldsymbol{\mu}^{(t)}_g$ be the means of $C^{(t)}$ and put $F(C,\boldsymbol{\mu})=\sum_g\sum_{i\in C_g}|\mathbf{x}_i-\boldsymbol{\mu}_g|^2$. Then

$$
W(C^{(t)})=F(C^{(t)},\boldsymbol{\mu}^{(t)})\ \geqslant\ F(C^{(t+1)},\boldsymbol{\mu}^{(t)})\ \geqslant\
F(C^{(t+1)},\boldsymbol{\mu}^{(t+1)})=W(C^{(t+1)}) ,
$$

the first inequality by the assignment half of Proposition 13.4(iii) and the second by the centre half, and the two equalities because $W(C)=\min_{\boldsymbol{\mu}}F(C,\boldsymbol{\mu})$ attained at the cluster means.

There are finitely many partitions of $n$ points into $K$ groups, and $W$ never increases, so the sequence $W(C^{(t)})$ is non-increasing and takes finitely many values, hence is eventually constant. Once $W(C^{(t+1)})=W(C^{(t)})$ both inequalities are equalities. By the second, and because the centre half of Proposition 13.4(iii) is minimised exactly at the cluster means, $\boldsymbol{\mu}^{(t+1)}=\boldsymbol{\mu}^{(t)}$. So the next assignment step faces the same means as the last one and, the tie rule depending only on the means, returns $C^{(t+1)}$ again. ∎

**Remark**. The theorem promises a fixed point of the algorithm and nothing more. It need not be a global minimum: the number of partitions into $K$ non-empty groups is a Stirling number of the second kind, which for $n=50$ and $K=3$ is $(3^{50}-3\cdot2^{50}+3)/6\approx1.2\cdot10^{23}$ (of the $3^{50}$ labellings, $3^{50}-3\cdot2^{50}+3$ use all three labels by inclusion–exclusion, and each partition arises from $3!$ of them), and no algorithm here searches them. The remedy, ISLR's, is to run Lloyd's algorithm from many random starts and keep the assignment with the smallest criterion; its Figure 12.9 shows six starts reaching three different values. Different starts can give different clusters, so two analysts with the same data and the same $K$ need not agree.

## 3 Hierarchical Clustering

> **Reading.** ISLR §12.4.2 (both reading lists). SRM learning outcome 5(c).

**Remark** (ISLR §12.4.2). Hierarchical clustering needs no $K$. Its bottom-up, or agglomerative, form builds a **dendrogram**: each leaf is an observation, and moving up, similar observations and groups fuse. The height of the fusion of two observations measures how different they are; their horizontal closeness in the picture means nothing, since the two branches at each fusion can be swapped. Cutting the dendrogram at a height gives clusters, and one dendrogram gives any number of clusters.

**Definition 13.7** (Agglomerative clustering and linkage; ISLR Algorithm 12.3, Table 12.3). Begin with $n$ singleton clusters and a dissimilarity $d(i,i')$. At each step merge the two clusters with the smallest **linkage** $D$, and record the merge height $D$. The usual linkages, for clusters $A$ and $B$, are

$$
\begin{align}
\text{single:}&\quad D(A,B)=\min_{i\in A,i'\in B}d(i,i') , \tag{2} \\
\text{complete:}&\quad D(A,B)=\max_{i\in A,i'\in B}d(i,i') , \tag{3} \\
\text{average:}&\quad D(A,B)=\frac1{|A||B|}\sum_{i\in A}\sum_{i'\in B}d(i,i') , \tag{4} \\
\text{centroid:}&\quad D(A,B)=|\bar{\mathbf{x}}_A-\bar{\mathbf{x}}_B| . \tag{5}
\end{align}
$$

The **dendrogram** draws each merge as a join at its height.

**Proposition 13.8** (Cutting the dendrogram). The clusters present at any moment form a partition, and every cluster at a later moment is a union of clusters at an earlier one. If the merge heights are non-decreasing, cutting the dendrogram at height $h$ therefore returns a partition, and the partitions obtained at different heights are nested.

**Proof.** The initial clusters partition the observations, and replacing two blocks by their union leaves a partition; each step does that once, which is also the nesting claim. When the merge heights are non-decreasing, the merges of height at most $h$ are the first $m$ merges for some $m$, so the cut at $h$ returns the clusters present after $m$ steps, a partition; and a lower cut takes fewer merges, so it returns an earlier state and refines a higher one. ∎

**Theorem 13.9** (No inversions). Suppose the linkage satisfies the **reducibility** inequality

$$
D(A\cup B,\,C)\ \geqslant\ \min\{D(A,C),\,D(B,C)\} \tag{6}
$$

for all disjoint clusters. Then the merge heights are non-decreasing, so the dendrogram has no **inversion**, a merge at a height below that of the previous merge. Single, complete and average linkage all satisfy (6).

**Proof.** Let $h$ be the height of the current merge, of $A$ with $B$, so $D(A,B)=h$ and every other pair of current clusters has linkage at least $h$. After the merge the available pairs are of two kinds. A pair not involving $A\cup B$ has the same linkage as before, hence at least $h$. A pair $(A\cup B,C)$ has, by (6), linkage at least $\min\{D(A,C),D(B,C)\}\geqslant h$, both terms having been available before the merge. So the next minimum is at least $h$.

Single linkage attains equality in (6), the minimum over $A\cup B$ being the smaller of the two minima. Complete linkage gives $\max\{D(A,C),D(B,C)\}\geqslant\min\{D(A,C),D(B,C)\}$. Average linkage gives $\{|A|D(A,C)+|B|D(B,C)\}/(|A|+|B|)$, a weighted mean of the two, hence at least their minimum. ∎

**Remark**. So the hypothesis of Proposition 13.8 holds for single, complete and average linkage, and their dendrograms can be cut at any height; the proof of Theorem 13.9 does not use that proposition.

**Remark**. Centroid linkage is not of this form: $\bar{\mathbf{x}}_{A\cup B}$ is a weighted mean of $\bar{\mathbf{x}}_A$ and $\bar{\mathbf{x}}_B$ and can be strictly closer to $\bar{\mathbf{x}}_C$ than either, so (6) fails and a merge can occur below the height of the merge that produced it. The dendrogram then crosses itself and cutting it at a height no longer returns the clusters the picture suggests. On `USArrests` there are five such inversions (Figure 13.4, right), and none for the other three linkages. (R's `hclust(d, "centroid")` computes (5) only when `d` holds squared Euclidean distances, so the computations below pass `dist(X)\^{}2` and take square roots of the heights.) ISLR's account: average and complete linkage are generally preferred to single, and centroid linkage, used in genomics, can produce an inversion, a fusion below either of the clusters fused.

**Proposition 13.10** (Single linkage chains). Under single linkage, two observations are in the same cluster at height $h$ if and only if they are joined by a chain $i=i_0,i_1,\ldots,i_r=i'$ with $d(i_{l-1},i_l)\leqslant h$ at every step.

**Proof.** Single linkage satisfies (6), so by Theorem 13.9 its merge heights are non-decreasing and, by Proposition 13.8, the clusters at height $h$ are those present after all merges of height at most $h$. Merging at height at most $h$ requires only one pair at distance at most $h$, so each cluster at height $h$ is connected in the graph on $\{1,\ldots,n\}$ whose edges are the pairs at distance at most $h$. Conversely, if $i\in A$ and $i'\in B$, two different clusters at height $h$, had $d(i,i')\leqslant h$, then $D(A,B)\leqslant h$, and the next merge, being of minimal linkage and, by Theorem 13.9, not below the previous ones, would be at height at most $h$, contrary to the cut. So the clusters at height $h$ are exactly the connected components of that graph, and membership of a component is the existence of such a chain. ∎

**Remark**. That is single linkage's characteristic behaviour and its characteristic failure. A sequence of intermediate points at small spacing joins two otherwise distant groups, so single linkage finds elongated clusters that the other linkages break, and one stray point between two clusters merges them. Complete linkage, which is governed by the farthest pair, produces compact clusters and splits elongated ones.

**Proposition 13.11** (Correlation-based distance; ISLR §12.4.2). Standardise each observation across its $p\geqslant2$ features, $\tilde x_{ij}=(x_{ij}-\bar x_i)/s_i$ with $\bar x_i$ and $s_i>0$ the mean and standard deviation (divisor $p-1$) of $x_{i1},\ldots,x_{ip}$, and let $r_{ii'}$ be the correlation between observations $i$ and $i'$ across features. Then

$$
|\tilde{\mathbf{x}}_i-\tilde{\mathbf{x}}_{i'}|^2=2(p-1)(1-r_{ii'}) , \tag{7}
$$

so on standardised observations the correlation-based dissimilarity $1-r_{ii'}$ is proportional to the squared Euclidean distance, and the two give the same hierarchical clustering under any linkage of Definition 13.7 except centroid.

**Proof.** Each $\tilde{\mathbf{x}}_i$ has entries summing to $0$ and $|\tilde{\mathbf{x}}_i|^2=p-1$, and $r_{ii'}=\tilde{\mathbf{x}}_i'\tilde{\mathbf{x}}_{i'}/(p-1)$. Expanding, $|\tilde{\mathbf{x}}_i-\tilde{\mathbf{x}}_{i'}|^2=|\tilde{\mathbf{x}}_i|^2+|\tilde{\mathbf{x}}_{i'}|^2-2\tilde{\mathbf{x}}_i'\tilde{\mathbf{x}}_{i'} =2(p-1)-2(p-1)r_{ii'}$. Multiplying every dissimilarity by the constant $2(p-1)$ multiplies the single, complete and average linkages by the same constant, so every comparison in the algorithm, and hence every merge, is unchanged. ∎

**Remark** (ISLR §12.4.2). Nothing here says which linkage or which dissimilarity is right, and there is no criterion to appeal to: the choices define what a cluster is. Euclidean distance on standardised columns and correlation-based dissimilarity answer different questions, the first about magnitudes and the second about profiles, and two observations can be close in one and far in the other. Variables measured in the same units, such as counts of the same kind, may be left unscaled when their differences in spread are meaningful, as ISLR notes in §12.2.4. ISLR's shoppers: Euclidean distance groups the infrequent shoppers together, correlation groups shoppers with similar preferences whatever their volume. Scaling matters in the same way: in ISLR's Figure 12.16 a shopper's purchases of socks and computers, counted in items, are dominated by socks; in dollars, by computers.

**Remark** (ISLR §12.4.2). The word **hierarchical** is an assumption. Proposition 13.8 forces the clusters at every moment of the algorithm to be nested, and ISLR's example is a population split evenly by sex and among three nationalities: the best two groups split by sex and the best three by nationality, and these are not nested. Then no single dendrogram returns both the best two and the best three clusters, and $K$-means, which solves each $K$ afresh, can do better for at least one of them.

## 4 Practical Issues in Clustering

> **Reading.** ISLR §12.4.3 (both reading lists). SRM learning outcomes 5(c), 5(d).

**Proposition 13.12** ($W$ cannot choose $K$). The optimal $W$ is non-increasing in $K$, and equals $0$ at $K=n$.

**Proof.** Any partition into $K$ groups is refined into one with $K+1$ by splitting a group with at least two members, and by Proposition 13.4(ii) applied within that group the within-cluster sum of squares cannot rise. At $K=n$ every cluster is a single point and each term of (1) vanishes. ∎

**Remark**. So the within-cluster criterion cannot choose $K$, and neither can anything else here: with no response there is nothing held out to predict. What is done is to plot the optimal $W$ against $K$ and look for an elbow, which is a judgement, or to check stability under resampling. When the clusters feed a model with a response, $K$ becomes a tuning parameter and cross-validation applies to that model.

**Remark** (ISLR §12.4.3). ISLR's **small decisions with big consequences**: whether to standardise, which dissimilarity, which linkage, where to cut, how many clusters. None has a right answer, and the advice is to try several and look for the patterns that persist. Nor is there a consensus on validating clusters, on telling real subgroups from clusters of noise; ISLR mentions methods that attach a $p$-value to a cluster.

**Remark** (ISLR §12.4.3). Both methods put every observation in a cluster, so a few outliers that belong to no group can distort all the clusters; mixture models give a soft version that need not. And clustering methods are not robust: clustering a subset of the data can give quite different clusters, so ISLR recommends clustering subsets as a check and reporting the results as a starting point for hypotheses, not as truth.

## 5 Computation in R

> **Reading.** ISLR §12.5.3–12.5.4 (labs). The principal components lab §12.5.1 is Unit 5's; §12.5.2 (matrix completion) is excluded by the syllabus. SRM learning outcomes 5(b)–5(d).

### 5.1 The ISLR labs

**Example 13.1** ($K$-means and hierarchical clustering on simulated data; ISLR §12.5.3). Fifty points in the plane, the first $25$ shifted by $(3,-4)$, ISLR's seeds:

In [ ]:
suppressMessages(library(ISLR2))
set.seed(2); x <- matrix(rnorm(50 * 2), ncol = 2)
x[1:25, 1] <- x[1:25, 1] + 3; x[1:25, 2] <- x[1:25, 2] - 4
km2 <- kmeans(x, 2, nstart = 20)
c(misplaced_by_K2 = min(sum(km2$cluster != rep(1:2, each = 25)),
                        sum(km2$cluster != rep(2:1, each = 25))))
set.seed(4); km3 <- kmeans(x, 3, nstart = 20); km3$size
table(cluster = km3$cluster, group = rep(1:2, each = 25))
set.seed(4)
c(nstart_1 = kmeans(x, 3, nstart = 1)$tot.withinss,
  nstart_20 = kmeans(x, 3, nstart = 20)$tot.withinss)
hcs <- lapply(c(complete = "complete", average = "average", single = "single"),
              function(m) hclust(dist(x), method = m))
sapply(hcs, function(h) table(factor(cutree(h, 2), levels = 1:2)))
table(cutree(hcs$single, 4))
table(complete = cutree(hcs$complete, 2), group = rep(1:2, each = 25))
table(scaled_complete = cutree(hclust(dist(scale(x))), 2), group = rep(1:2, each = 25))
x3 <- matrix(rnorm(30 * 3), ncol = 3); dc <- as.dist(1 - cor(t(x3)))
range(dc)

$K=2$ recovers the two groups exactly. With $K=3$ the third cluster takes points from both groups ($8$ and $2$). With one start the criterion $W$, which `kmeans` reports as `tot.withinss`, is $104.33$; with twenty it is $97.98$, ISLR's two numbers and the reason for many starts (Theorem 13.6). Cut at two clusters, complete linkage gives $25$ and $25$, average linkage $28$ and $22$, and single linkage puts one point alone; cut at four, single linkage still leaves two singletons. The correlation-based dissimilarity needs at least three features to be informative and lies in $[0,2]$.

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 2.6)
op <- par(mfrow = c(1, 3), mar = c(0.5, 3, 2, 0.5), cex = 0.6)
plot(hcs$complete, labels = FALSE, main = "complete linkage", xlab = "", sub = "")
plot(hclust(dist(scale(x))), labels = FALSE, main = "scaled features", xlab = "", sub = "")
plot(hclust(dc), labels = FALSE, main = "correlation distance", xlab = "", sub = "")
par(op)

*Figure 13.1.* Example 13.1, the dendrograms of ISLR's lab §12.5.3: complete linkage on the fifty points, the same after scaling each feature to standard deviation one, and complete linkage on the correlation-based dissimilarity of thirty points in three dimensions.

**Remark**. In Figure 13.1 the two-cluster cut recovers the two groups exactly both before and after scaling: the shift of $3$ and $-4$ is large against the unit noise in each feature, so the choice of units does not matter here. The correlation-based dendrogram has no such structure, the thirty points being pure noise.

**Example 13.2** (The `NCI60` cancer cell lines; ISLR §12.5.4). Expression of $6830$ genes in $64$ cancer cell lines of $14$ labelled types. The labels are not used by the methods, only to judge them afterwards.

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3)
nci.labs <- NCI60$labs; nci.data <- NCI60$data
pr.nci <- prcomp(nci.data, scale = TRUE)
pve <- 100 * pr.nci$sdev^2 / sum(pr.nci$sdev^2)
round(rbind(pve = pve[1:7], cumulative = cumsum(pve)[1:7]), 1)
sd.data <- scale(nci.data); hc.out <- hclust(dist(sd.data))
hc.clusters <- cutree(hc.out, 4)
table(hc.clusters, nci.labs)[, c("BREAST", "COLON", "LEUKEMIA", "MELANOMA", "RENAL")]
set.seed(2); km.clusters <- kmeans(sd.data, 4, nstart = 20)$cluster
table(km.clusters, hc.clusters)
table(pc_clusters = cutree(hclust(dist(pr.nci$x[, 1:5])), 4), hc.clusters)

The first seven components explain $38.5\%$ of the variance and the scree plot has an elbow there, ISLR's reading. Complete linkage on the standardised data, cut at four clusters, puts all six leukemia lines in cluster $3$ and spreads the breast lines over three clusters. $K$-means with $K=4$ agrees on the leukemia cluster and splits hierarchical cluster $1$ three ways, as in ISLR. Clustering the first five score vectors instead of the full data gives yet another partition: the components are a denoised version of the data, and a different input gives different clusters.

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 2.9)
cols <- rainbow(length(unique(nci.labs)))[as.numeric(as.factor(nci.labs))]
op <- par(mfrow = c(1, 2), mar = c(4, 4.2, 0.8, 0.8), cex = 0.85)
plot(pr.nci$x[, 1:2], col = cols, pch = 19, xlab = "Z1", ylab = "Z2")
plot(pr.nci$x[, c(1, 3)], col = cols, pch = 19, xlab = "Z1", ylab = "Z3")
par(op)

*Figure 13.2.* Example 13.2, ISLR's Figure 12.17: the first three principal component score vectors of the `NCI60` cell lines, coloured by cancer type. Lines of one type tend to lie near each other, so the types are partly visible without the labels.

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 2.8)
op <- par(mfrow = c(1, 2), mar = c(0.5, 3, 2, 0.5), cex = 0.55)
plot(hc.out, labels = FALSE, main = "complete linkage", xlab = "", sub = "")
abline(h = 139, col = "#B45309", lwd = 2, lty = 2)
plot(hclust(dist(sd.data), method = "single"), labels = FALSE, main = "single linkage",
     xlab = "", sub = "")
par(op)

*Figure 13.3.* Example 13.2 (ISLR's Figure 12.19 and its lab): complete linkage on the standardised `NCI60` data with the lab's cut at height $139$, which gives the four clusters of the example, and single linkage on the same dissimilarities.

**Remark**. In Figure 13.3 complete linkage gives the more balanced tree; single linkage trails, merging the cell lines one at a time, the chaining of Proposition 13.10. The cut at height $139$ returns the same four clusters as `cutree(hc.out, 4)`, of sizes $40$, $7$, $8$ and $9$.

**Remark**. Figure 13.2 is the principal components view of Example 13.2: the first three score vectors, along which the cell lines are most spread out (Unit 5's theorem **principal components are eigenvectors**). ISLR draws it before clustering, to see whether groups are visible at all.

### 5.2 The `USArrests` data

The checks below use `USArrests`, four crime and population measurements on the fifty states (three arrest rates per $100{,}000$ residents and a percentage), standardised, the scaling Unit 5's example `USArrests` used.

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3)
X <- scale(USArrests); n <- nrow(X); k <- ncol(X)
round(rbind(mean = colMeans(USArrests), sd = apply(USArrests, 2, sd)), 2)

### 5.3 $K$-means (§2)

In [ ]:
set.seed(4); km <- kmeans(X, centers = 3, nstart = 50)
wss <- function(lab) sum(sapply(unique(lab), function(g) {
  Z <- X[lab == g, , drop = FALSE]; sum(scale(Z, scale = FALSE)^2) }))
g1 <- X[km$cluster == 1, ]
c(package_tot_withinss = km$tot.withinss, manual = wss(km$cluster),
  total = sum(scale(X, scale = FALSE)^2),
  within_plus_between = km$tot.withinss + km$betweenss,
  pairwise_form = sum(as.matrix(dist(g1))^2)/nrow(g1),
  twice_within_cluster_1 = 2*sum(scale(g1, scale = FALSE)^2))

In [ ]:
lloyd <- function(Xm, K, seed) { set.seed(seed)
  lab <- sample(K, nrow(Xm), replace = TRUE); tr <- numeric(0)
  for (it in 1:30) {
    C <- t(sapply(1:K, function(g) colMeans(Xm[lab == g, , drop = FALSE])))
    D <- sapply(1:K, function(g) rowSums(sweep(Xm, 2, C[g, ])^2))
    lab <- max.col(-D, ties.method = "first"); tr <- c(tr, sum(apply(D, 1, min))) }
  tr }
round(lloyd(X, 3, 11)[1:8], 3)

The sequence is non-increasing and constant from the sixth round, as Theorem 13.6 requires. It is a fixed point and not the global minimum: repeated single starts land in several places.

In [ ]:
set.seed(6)
one <- replicate(200, kmeans(X, centers = 3, nstart = 1)$tot.withinss)
c(best = min(one), worst = max(one), distinct_values = length(unique(round(one, 6))),
  proportion_at_best = mean(round(one, 6) == round(min(one), 6)),
  with_nstart_50 = kmeans(X, centers = 3, nstart = 50)$tot.withinss)

In [ ]:
set.seed(7)
t(sapply(1:8, function(K) c(K = K,
  within = kmeans(X, centers = K, nstart = 25)$tot.withinss)))

### 5.4 Hierarchical clustering (§3)

In [ ]:
dd <- dist(X)
t(sapply(c("single", "complete", "average", "centroid"), function(m) {
  h <- hclust(if (m == "centroid") dd^2 else dd, method = m)
  if (m == "centroid") h$height <- sqrt(h$height)   # eq:centroid needs squared distances
  c(smallest_height_step = min(diff(h$height)),
    inversions = sum(diff(h$height) < 0),
    largest_height = max(h$height)) }))

In [ ]:
hc <- hclust(dd, method = "complete")
tab <- table(cut2 = cutree(hc, 2), cut4 = cutree(hc, 4))
tab
c(each_four_cluster_lies_in_one_two_cluster = all(colSums(tab > 0) == 1))

In [ ]:
hs <- hclust(dd, method = "single"); h0 <- hs$height[n - 4]
lab <- cutree(hs, h = h0 + 1e-9)
A <- as.matrix(dd) <= h0 + 1e-9
comp <- rep(0, n); cc <- 0
for (i in 1:n) if (comp[i] == 0) { cc <- cc + 1; front <- i
  while (length(front)) { comp[front] <- cc
    front <- setdiff(which(apply(A[front, , drop = FALSE], 2, any)),
                     which(comp > 0)) } }
c(height = h0, clusters_from_cutree = length(unique(lab)),
  connected_components = cc,
  same_partition = all(table(lab, comp) %*% rep(1, cc) ==
                       apply(table(lab, comp), 1, max)))

In [ ]:
op <- par(mfrow = c(1, 2), mar = c(0.6, 3.6, 2.0, 0.6), cex = 0.55,
          mgp = c(2.2, 0.7, 0))
plot(hclust(dd, "complete"), labels = abbreviate(rownames(USArrests), 4),
     main = "complete linkage", xlab = "", sub = "", ylab = "height")
abline(h = mean(hclust(dd, "complete")$height[c(n - 4, n - 3)]), col = "#B45309", lty = 2,
       lwd = 2)
hcen <- hclust(dd^2, "centroid"); hcen$height <- sqrt(hcen$height)
plot(hcen, labels = abbreviate(rownames(USArrests), 4),
     main = "centroid linkage: five inversions", xlab = "", sub = "",
     ylab = "height")
par(op)

*Figure 13.4.* Left: a dendrogram with non-decreasing heights (Theorem 13.9), cut at the dashed line to give four clusters. Right: centroid linkage on the same data, (5), where a merge can sit below the merge that produced it, so the branches double back and a horizontal cut no longer reads off the clusters drawn.

### 5.5 The two methods on the same data

In [ ]:
set.seed(9)
km3 <- kmeans(X, centers = 3, nstart = 50)$cluster
hc3 <- cutree(hclust(dd, "complete"), 3)
table(kmeans = km3, complete_linkage = hc3)
c(within_kmeans = wss(km3), within_hclust = wss(hc3))

$K$-means minimises (1) directly and hierarchical clustering does not, so the first reaches the smaller value, $78.32$ against $84.40$. The partitions differ substantially: complete linkage puts $31$ states in one cluster, which $K$-means splits $17$, $13$ and $1$, and $K$-means joins all of the other two complete-linkage clusters into one. Neither is wrong; they answer different criteria.

### 5.6 Clustering in practice

#### Not standardising, or standardising without asking

Unit 5's proposition **scaling changes the components** showed the first component of `USArrests` changing from `Assault` alone to a balanced combination. The same choice governs clustering, because every dissimilarity in Definition 13.7 is computed in whatever units the columns carry.

In [ ]:
cl_raw <- cutree(hclust(dist(USArrests), "complete"), 3)
cl_std <- cutree(hclust(dist(X), "complete"), 3)
table(unstandardised = cl_raw, standardised = cl_std)
c(states_in_the_diagonal_cells = sum(diag(table(cl_raw, cl_std))),
  n = nrow(USArrests))

The cluster labels are arbitrary, so the table is the object to read and not the diagonal: whichever way the three groups of one analysis are matched to the three of the other, a large minority of the fifty states changes group. The two analyses are not the same analysis.

The symptom is a clustering dominated by whichever variable happens to be recorded on the largest scale. Standardise unless the columns are already commensurable, and say which was done. Neither choice is right in general: standardising declares all four variables equally important, and not standardising declares that the assault rate matters more than the murder rate because it varies over a wider range. Columns in one unit need not be commensurable in spread.

#### Choosing $K$ by the within-cluster criterion

The symptom is a $K$ chosen as the minimiser of the within-cluster sum of squares, which by Proposition 13.12 is always $K=n$; §5.3 prints the sequence falling monotonically to $K=8$ with no sign of stopping. What is available instead is set out in the remark after Proposition 13.12.

#### Reporting one run of $K$-means

In [ ]:
set.seed(12)
r <- t(replicate(60, { z <- kmeans(X, centers = 4, nstart = 1)
  c(z$tot.withinss, length(unique(z$cluster))) }))
c(best = min(r[, 1]), worst = max(r[, 1]),
  spread_percent = 100*(max(r[, 1]) - min(r[, 1]))/min(r[, 1]),
  distinct_optima = length(unique(round(r[, 1], 6))))

The symptom is a $K$-means result reported without `nstart`, or two analysts getting different clusters from the same data. Theorem 13.6 guarantees a fixed point and nothing more, and the starts here reach several distinct values spanning a wide range. Use many starts and report the number used.

#### Cutting a dendrogram that has inversions

In [ ]:
hcen <- hclust(dd^2, "centroid"); hcen$height <- sqrt(hcen$height)
bad <- which(diff(hcen$height) < 0)
rbind(merge_index = bad[1:3],
      height_before = round(hcen$height[bad[1:3]], 4),
      height_after = round(hcen$height[bad[1:3] + 1], 4))
c(total_inversions = sum(diff(hcen$height) < 0))
list(cut_by_height = tryCatch(length(unique(cutree(hcen,
       h = hcen$height[bad[1]]))), error = function(e) conditionMessage(e)),
     cut_by_number = length(unique(cutree(hcen, k = 3))))

R declines to cut by height at all, which is the honest response: with the heights not sorted there is no partition for a horizontal line to name. Cutting by number of clusters still works, because that reads the merge order rather than the heights.

The symptom is a dendrogram whose branches double back, and a horizontal cut that either fails or returns a different number of clusters from the one the picture suggests. Centroid linkage fails the reducibility inequality (6) of Theorem 13.9, so its heights are not monotone and the usual reading of a cut is not available. Use single, complete or average linkage, all three of which the theorem covers, or cut by number of clusters rather than by height.

## 6 Exercises

**Exercise 13.1** (ISLR §12.6, exercise 1). (a) Prove ISLR's (12.18): $|C_k|^{-1}\sum_{i,i'\in C_k}\sum_j(x_{ij}-x_{i'j})^2 =2\sum_{i\in C_k}\sum_j(x_{ij}-\bar x_{kj})^2$. (b) On the basis of this identity, argue that Algorithm 12.2 decreases the objective (12.17) at each iteration.

**Solution.** (a) Multiply Proposition 13.4(i) by $2$; the proof there expands the double sum. (b) By (a) the objective (12.17) is $2W(C)$. Theorem 13.6 shows that neither half of an iteration increases $W$: replacing the centres by the cluster means minimises over centres for the current partition, and reassigning each observation to its nearest mean minimises over partitions for the current centres (Proposition 13.4(iii)). ■

**Exercise 13.2** (ISLR §12.6, exercise 2). Four observations have dissimilarities $d_{12}=0.3$, $d_{13}=0.4$, $d_{14}=0.7$, $d_{23}=0.5$, $d_{24}=0.8$, $d_{34}=0.45$. (a) Sketch the complete linkage dendrogram, with the fusion heights. (b) The same for single linkage. (c), (d) Cut each into two clusters. (e) Can the dendrogram be drawn with the leaves in another order without changing its meaning?

**Solution.**

In [ ]:
d <- as.dist(matrix(c(0, .3, .4, .7,  .3, 0, .5, .8,
                     .4, .5, 0, .45,  .7, .8, .45, 0), 4))
hc <- hclust(d, "complete"); hs <- hclust(d, "single")
list(complete_heights = hc$height, single_heights = hs$height,
     complete_2 = cutree(hc, 2), single_2 = cutree(hs, 2))

(a) Complete linkage fuses $\{1,2\}$ at $0.3$, then $\{3,4\}$ at $0.45$, then the two at $0.8$, the largest of $0.4,0.7,0.5,0.8$. (b) Single linkage fuses $\{1,2\}$ at $0.3$, then $3$ with them at $\min\{0.4,0.5\}=0.4$, then $4$ at $\min\{0.7,0.8,0.45\}=0.45$. (c) $\{1,2\}$ and $\{3,4\}$. (d) $\{1,2,3\}$ and $\{4\}$. (e) Yes: the two branches at every fusion can be swapped, for example $1,2$ to $2,1$, without changing any fusion or height; only the heights carry meaning. ■

**Exercise 13.3** (ISLR §12.6, exercise 3). Perform $K$-means with $K=2$ by hand on the six observations $(1,4)$, $(1,3)$, $(0,4)$, $(5,1)$, $(6,2)$, $(4,0)$: assign clusters at random, then alternate between computing the centroids and reassigning, until the assignment stops changing.

**Solution.**

In [ ]:
X3 <- cbind(c(1, 1, 0, 5, 6, 4), c(4, 3, 4, 1, 2, 0)); set.seed(1)
lab <- sample(2, 6, replace = TRUE); lab
repeat {
  C3 <- t(sapply(1:2, function(g) colMeans(X3[lab == g, , drop = FALSE])))
  new <- max.col(-sapply(1:2, function(g) rowSums(sweep(X3, 2, C3[g, ])^2)),
                 ties.method = "first")
  print(list(centroids = C3, assignment = new))
  if (all(new == lab)) break
  lab <- new }

The random start is $1,2,1,1,2,1$, with centroids $(2.5,2.25)$ and $(3.5,2.5)$. Reassigning puts the first three observations in cluster $1$ and the last three in cluster $2$; the new centroids are $(2/3,11/3)$ and $(5,1)$, and a further reassignment changes nothing. The final clusters are $\{(1,4),(1,3),(0,4)\}$ and $\{(5,1),(6,2),(4,0)\}$. ■

**Exercise 13.4** (ISLR §12.6, exercise 4). Single and complete linkage dendrograms are built from the same data. (a) At some point $\{1,2,3\}$ and $\{4,5\}$ fuse in both. Which fusion is higher, or is there not enough information? (b) At some point $\{5\}$ and $\{6\}$ fuse in both. Which fusion is higher?

**Solution.** (a) Not enough information to say which is strictly higher. The complete linkage of the two clusters, $\max d(i,i')$ over the six cross pairs, is at least the single linkage, the minimum over the same pairs, so the complete linkage fusion is at least as high; they are equal when all six distances are equal. (b) The same height: for two single observations both linkages equal $d(5,6)$. ■

**Exercise 13.5** (ISLR §12.6, exercise 5). ISLR's Figure 12.16 records eight shoppers' purchases of socks and computers: shoppers $1,\ldots,8$ bought $8,11,7,6,5,6,7,8$ pairs of socks and $0,0,0,0,1,1,1,1$ computers. In dollars (the figure's right panel) each computer bought is well over \$1500 and the socks a few dollars. Describe the $K$-means clusters with $K=2$ on the raw counts, on the counts each scaled to standard deviation one, and on the dollars spent.

**Solution.** The squared Euclidean distance $\sum_j(x_{ij}-x_{i'j})^2$ is dominated by the variable with the larger spread. On the raw counts the number of socks varies far more than the number of computers, so the two clusters split the shoppers by how many socks they bought. Scaled, both variables count equally, and the split reflects socks and computers together; with these data, whether a computer was bought separates the shoppers. (On the raw counts the optimal split in fact isolates shopper $2$, the one with $11$ pairs of socks.) In dollars a computer outweighs any number of socks bought here by well over an order of magnitude, so the clusters split the shoppers by whether they bought a computer. The three analyses answer three different questions, as the remark on dissimilarities after Proposition 13.11 says. ■

**Exercise 13.6** (ISLR §12.6, exercise 7). Show that if each observation has been centred to mean zero and scaled to standard deviation one across its features, then $1-r_{ii'}$ is proportional to the squared Euclidean distance between observations $i$ and $i'$; check it on `USArrests`.

**Solution.** This is Proposition 13.11, with constant $2(p-1)$:

In [ ]:
Xr <- t(scale(t(USArrests))); r <- cor(t(Xr)); D2 <- as.matrix(dist(Xr))^2
range((D2 / (1 - r))[upper.tri(D2)])

The ratio is $6=2(4-1)$ for every pair of states. ■

**Exercise 13.7** (ISLR §12.6, exercise 9). For `USArrests`: (a) cluster the states by complete linkage and Euclidean distance; (b) cut into three clusters and list them; (c) repeat after scaling the variables to standard deviation one; (d) what does scaling change, and should the variables be scaled?

**Solution.**

In [ ]:
c3 <- cutree(hclust(dist(USArrests), "complete"), 3); table(c3)
split(names(c3), c3)
s3 <- cutree(hclust(dist(scale(USArrests)), "complete"), 3)
table(raw = c3, scaled = s3)

(a), (b) Unscaled, the three clusters have $16$, $14$ and $20$ states; the first holds the states with high assault rates, from Alabama to South Carolina; the second ($14$ states, from Arkansas to Wyoming) has intermediate rates; the third ($20$ states, from Connecticut to Wisconsin) holds the low-crime states. (c) Scaled, the partition changes: of the $16$ states in the first raw cluster, $9$ move to one scaled cluster, $6$ to another and $1$ to the third, and only the third raw cluster survives intact inside one scaled cluster. (d) Unscaled, `Assault`, with standard deviation $83$ against `Murder`'s $4.4$, decides almost every distance, so the clustering is essentially of the assault rate. The four variables are in different units (arrests per $100{,}000$ and a percentage), so they should be scaled unless there is a reason to let assaults dominate. ■

**Exercise 13.8** (ISLR §12.6, exercise 10). (a) Simulate $60$ observations of $50$ variables in three classes of $20$ with a mean shift between classes. (b) Plot the first two principal component score vectors. (c) Run $K$-means with $K=3$ and compare with the true classes; (d) with $K=2$; (e) with $K=4$; (f) with $K=3$ on the first two score vectors; (g) with $K=3$ after scaling each variable.

**Solution.**

In [ ]:
options(repr.plot.width = 4.2, repr.plot.height = 2.8)
set.seed(2); x10 <- matrix(rnorm(60 * 50), ncol = 50); y10 <- rep(1:3, each = 20)
x10[1:20, 1:2] <- x10[1:20, 1:2] + 4; x10[21:40, 1] <- x10[21:40, 1] - 4
x10[41:60, 2] <- x10[41:60, 2] - 4
pr10 <- prcomp(x10); set.seed(1)
fits <- list(K3 = kmeans(x10, 3, nstart = 20), K2 = kmeans(x10, 2, nstart = 20),
             K4 = kmeans(x10, 4, nstart = 20),
             K3_scores = kmeans(pr10$x[, 1:2], 3, nstart = 20),
             K3_scaled = kmeans(scale(x10), 3, nstart = 20))
lapply(fits, function(z) table(cluster = z$cluster, class = y10))
par(mar = c(4, 4.2, 0.8, 0.8), cex = 0.85)
plot(pr10$x[, 1:2], col = c("#1E3A8A", "#B45309", "#15803D")[y10], pch = 19)

(a) The three classes differ in the means of the first two variables only. (b) The first two score vectors show three well-separated groups. (c) $K=3$ recovers the classes exactly (the labels are arbitrary). (d) $K=2$ keeps class $1$ and merges classes $2$ and $3$. (e) $K=4$ splits class $1$ into $11$ and $9$. (f) On the two score vectors $K=3$ again recovers the classes exactly: they carry the signal and leave out most of the noise. (g) Scaled, the clusters no longer match the classes. The shift raises the standard deviation of the two informative variables, so scaling shrinks the separation between classes while leaving the $48$ noise variables at full size. ■

**Exercise 13.9** (ISLR §12.6, exercise 13). ISLR's `Ch12Ex13.csv` holds the expression of $1000$ genes in $40$ tissue samples, the first $20$ healthy and the last $20$ diseased. (b) Cluster the samples hierarchically with correlation-based distance. Do the two groups separate? Does the result depend on the linkage? (c) Which genes differ most between the two groups?

**Solution.**

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3)
G <- read.csv("data/islr/Ch12Ex13.csv", header = FALSE); dim(G)
dG <- as.dist(1 - cor(G)); grp <- rep(c("healthy", "diseased"), each = 20)
lapply(c(complete = "complete", average = "average", single = "single",
         centroid = "centroid"), function(m) table(cutree(hclust(dG, m), 2), grp))
tt <- apply(G, 1, function(g) t.test(g[1:20], g[21:40])$statistic)
blk <- c(11:20, 501:600)
c(abs_t_over_4_in_11_20 = sum(abs(tt[11:20]) > 4),
  in_501_600 = sum(abs(tt[501:600]) > 4),
  largest_abs_t_elsewhere = max(abs(tt[-blk])), mean_t_in_blocks = mean(tt[blk]),
  min_abs_t_in_blocks = min(abs(tt[blk])), max_t_in_blocks = max(tt[blk]))

(b) Partly, and it depends on the linkage. Complete and average linkage, cut at two, put all $20$ diseased samples in one cluster with about half the healthy ones; single and centroid linkage leave one healthy sample on its own and put everything else together. (c) For each gene a two-sample $t$ statistic compares the groups. Two blocks stand out: every one of genes $11$ to $20$ and $501$ to $600$ has $|t|>4.1$, every other gene $|t|<4$, and the statistics in the blocks are negative, the genes more highly expressed in the diseased samples. The clustering uses all $1000$ genes, $890$ of them noise, which is why it separates the groups only partly. ■

**Exercise 13.10** (SOA sample question 1). For $x_1=(-1,0)$, $x_2=(1,1)$, $x_3=(2,-1)$, $x_4=(5,10)$, hierarchical clustering with complete linkage and Euclidean distance: calculate the intercluster dissimilarity between $\{x_1,x_2\}$ and $\{x_4\}$. (A) $2.2$; (B) $3.2$; (C) $9.9$; (D) $10.8$; (E) $11.7$.

**Solution.** Complete linkage (3) takes the largest of $d(x_1,x_4)=\sqrt{136}=11.66$ and $d(x_2,x_4)=\sqrt{97}=9.85$: $11.66$, (E). ■

**Exercise 13.11** (SOA sample question 2). Which are true? I. The number of clusters must be pre-specified for both $K$-means and hierarchical clustering. II. $K$-means is less sensitive to outliers than hierarchical clustering. III. $K$-means requires random assignments while hierarchical clustering does not. (A) I only; (B) II only; (C) III only; (D) I, II and III; (E) none of (A)–(D).

**Solution.** I is false: hierarchical clustering needs no $K$ (Definition 13.7). II is false: both put every observation in a cluster, so outliers distort both. III is true: Lloyd's algorithm starts from a random assignment (Definition 13.5), and agglomerative clustering is deterministic. (C). ■

**Exercise 13.12** (SOA sample question 15). Twelve points are initially assigned to three clusters: A: $(2,-1)$, $(-1,2)$, $(-2,1)$, $(1,2)$; B: $(4,0)$, $(4,-1)$, $(0,-2)$, $(0,-5)$; C: $(-1,0)$, $(3,8)$, $(-2,0)$, $(0,0)$. After one iteration of $K$-means with Euclidean distance, how many points are in the smallest cluster? (A) $0$; (B) $1$; (C) $2$; (D) $3$; (E) $4$.

**Solution.**

In [ ]:
P <- rbind(c(2, -1), c(-1, 2), c(-2, 1), c(1, 2), c(4, 0), c(4, -1), c(0, -2),
           c(0, -5), c(-1, 0), c(3, 8), c(-2, 0), c(0, 0))
g0 <- rep(c("A", "B", "C"), each = 4)
cen <- apply(P, 2, function(v) tapply(v, g0, mean)); cen
table(c("A", "B", "C")[max.col(-as.matrix(dist(rbind(cen, P)))[-(1:3), 1:3])])

The centroids are $(0,1)$, $(2,-2)$ and $(0,2)$; reassigning each point to its nearest centroid (Definition 13.5) leaves $4$, $5$ and $3$ points. (D). ■

**Exercise 13.13** (SOA sample question 16). Which statement applies to $K$-means clustering and not to hierarchical clustering? (A) Two people performing one iteration on the same data get the same result. (B) At each iteration the number of clusters exceeds that of the previous iteration. (C) The algorithm needs to be run only once, however many clusters are finally used. (D) The algorithm must be initialised with an assignment of the data points to clusters. (E) None of (A)–(D).

**Solution.** (A) fails for $K$-means, whose start is random. (B) holds for neither: $K$-means keeps $K$ clusters, and agglomerative clustering decreases their number. (C) is true of hierarchical clustering, one dendrogram serving every $K$ (Proposition 13.8), not of $K$-means. (D) is Definition 13.5; hierarchical clustering starts from singletons. (D). ■

**Exercise 13.14** (SOA sample question 32). For $n$ observations with $p$ features, which are true? I. The observations can be clustered on the basis of the features to find subgroups of observations. II. The features can be clustered on the basis of the observations to find subgroups of features. III. Clustering is an unsupervised method, often part of exploratory data analysis. (A) none; (B) I and II only; (C) I and III only; (D) II and III only; (E) none of (A)–(D).

**Solution.** All three are true: II by transposing $\mathbf{X}$, as the remark at the start of §2 says, and III by the remark at the start of §1. So the answer is (E). ■

**Exercise 13.15** (SOA sample question 34). Which are true? I. With $K$ fixed, $K$-means always produces the same assignments. II. Given a linkage and a dissimilarity, hierarchical clustering always produces the same assignments for a given number of clusters. III. Cutting a dendrogram into five clusters gives the same assignments as $K$-means with $K=5$. (A) I only; (B) II only; (C) III only; (D) I, II and III; (E) none of (A)–(D).

**Solution.** I is false: the start is random and different starts reach different fixed points (the remark after Theorem 13.6). II is true: the algorithm of Definition 13.7 is deterministic (ties aside). III is false: the two methods optimise different things, and §5.5 shows them disagreeing on `USArrests`. (B). ■

**Exercise 13.16** (SOA sample question 36). Which are true of hierarchical clustering? I. It may not assign extreme outliers to any cluster. II. The dendrogram can be used to obtain different numbers of clusters. III. It is not robust to small changes in the data. (A) none; (B) I and II only; (C) I and III only; (D) II and III only; (E) none of (A)–(D).

**Solution.** I is false: every cut by number of clusters, and every height cut of a dendrogram without inversions, is a partition (Proposition 13.8), so every observation is in a cluster. II is true by the same proposition. III is true: ISLR's remark on robustness in §4. (D). ■

**Exercise 13.17** (SOA sample question 40). Which are true? I. Cutting a dendrogram at a lower height will not decrease the number of clusters. II. $K$-means requires plotting the data before choosing the number of clusters. III. For a given number of clusters, hierarchical clustering can sometimes give less accurate results than $K$-means. (A) none; (B) I and II only; (C) I and III only; (D) II and III only; (E) none of (A)–(D).

**Solution.** I is true: for a dendrogram without inversions (single, complete and average linkage; the remark after Theorem 13.9) the partitions are nested (Proposition 13.8), and a lower cut refines a higher one, so it has at least as many clusters. II is false: nothing in Definition 13.5 needs a plot. III is true: when the best clusters are not nested, no dendrogram finds the best partition for every $K$, ISLR's sex-and-nationality example in §3. (C). ■

**Exercise 13.18** (SOA sample question 43). Which is NOT true of clustering? (A) It is used to discover structure in a data set. (B) It finds homogeneous subgroups among the observations. (C) It is unsupervised. (D) It reduces the dimensionality of a data set while retaining a good fraction of the variance. (E) In $K$-means the number of clusters must be pre-specified.

**Solution.** (D) describes principal components (Unit 5's proposition **proportion of variance explained**), not clustering. (D). ■

**Exercise 13.19** (SOA sample question 59). Five observations $(1,3)$, $(0,4)$, $(6,2)$ in cluster $1$ and $(5,2)$, $(1,6)$ in cluster $2$: calculate the total within-cluster variation of this assignment, by Euclidean distance. (A) $32.0$; (B) $70.3$; (C) $77.3$; (D) $118.3$; (E) $141.0$.

**Solution.** The within-cluster variation is ISLR's (12.16), which by the remark after Proposition 13.4 (via (i)) is $2W(C)$:

In [ ]:
P59 <- rbind(c(1, 3), c(0, 4), c(6, 2), c(5, 2), c(1, 6)); g59 <- c(1, 1, 1, 2, 2)
W59 <- sum(sapply(1:2, function(g) sum(scale(P59[g59 == g, ], scale = FALSE)^2)))
c(W = W59, ISLR_variation = 2 * W59)

$W=22.67+16=38.67$, and the variation is $77.33$, (C). The answer $38.7$, the plain sum of squares about the centroids, is not among the choices: the question uses ISLR's factor $2$. ■

**Exercise 13.20** (SOA sample question 60). Which are true of choosing $K$ in $K$-means? I. Set $K=n$. II. Choose $K$ to minimise the total within-cluster variation. III. The choice of $K$ is subjective, and no one method determines the optimal number. (A) I only; (B) II only; (C) III only; (D) I, II and III; (E) none of (A)–(D).

**Solution.** By Proposition 13.12 the optimal within-cluster variation is non-increasing in $K$ and $0$ at $K=n$, so II leads to I, which makes every observation its own cluster and says nothing. III is the remark after that proposition. (C). ■